# RAG

## **0. Imports**

In [1]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
!pip install langchain_community > /dev/null 2>&1
!pip install FlagEmbedding > /dev/null 2>&1 # Embedding model
!pip install faiss-cpu > /dev/null 2>&1 # Faiss
!pip install -U langchain langchain-core langchain-ollama > /dev/null 2>&1 # Ollama
!pip install colab-xterm > /dev/null 2>&1 # terminal to pull models with ollama

In [10]:

# Basic
import pandas as pd
import numpy as np
import re
import math
from typing import List, Optional
from collections import defaultdict
import random
import json
import warnings
warnings.filterwarnings("ignore")

#Langchain
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings

# Ollama for LLMs
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Torch and HF
import torch
from huggingface_hub import login
hf_token = "" # insert you hf token
login()

# Metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report


#BGEM3
from FlagEmbedding import BGEM3FlagModel

# Transformers
from transformers import pipeline

# WORKING DIRECTORY
wd = "" # Change to your WD

In [4]:
# Create an embedding object compatible with langchain embeddings

class BGEM3Embeddings(Embeddings):
    def __init__(self, model_name = "BAAI/bge-m3", device = None, normalize = True):

        self.model = BGEM3FlagModel(model_name, use_fp16=True, device=device)
        self.normalize = normalize

    def _encode(self, texts: List[str]):

        out = self.model.encode(
            texts,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
            batch_size=64,
            max_length=8192,
        )["dense_vecs"]
        vecs = np.array(out, dtype=np.float32)
        if self.normalize:
            norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
            vecs = vecs / norms
        return vecs

    def embed_documents(self, texts):
        return self._encode(texts).tolist()

    def embed_query(self, text):
        return self._encode([text]).tolist()[0]

## **1. Data Preparation**

In [11]:
# Read main df
df = pd.read_csv(wd + "final_df.csv")
df.head()

,spotify_id,input_title,input_artist,genius_title,genius_artist,url,lyrics,year,release_date,genre,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,isrc
0,46XOHK5QfCC83yg5SOGjZk,Bottom,TOOL,Bottom,Tool,https://genius.com/Tool-bottom-lyrics,42 Contributors\nTranslations\nTürkçe\nРусский...,1993,1993-04-06,metal,...,0.697,0.8880,2.0,0.125,-7.278,1.0,0.0348,139.510,0.278,USVR10000012
1,4Wvnm0J3Zz9cmMdEMMghGA,Undertow,TOOL,Undertow,Tool,https://genius.com/Tool-undertow-lyrics,31 Contributors\nTranslations\nTürkçe\nUnderto...,1993,1993-04-06,metal,...,0.797,0.8050,2.0,0.278,-6.623,1.0,0.0538,171.523,0.474,USVR10900085
2,2cF1W1G0sERJu0Y49tGjnx,Intolerance,TOOL,Intolerance,Tool,https://genius.com/Tool-intolerance-lyrics,36 Contributors\nTranslations\nTürkçe\nIntoler...,1993,1993-04-06,metal,...,0.941,0.2860,7.0,0.159,-7.566,1.0,0.0689,101.892,0.205,USVR10000009
3,0QRxtcxL31dRAeiUUuENPu,Territory,Sepultura,Territory,Sepultura,https://genius.com/Sepultura-territory-lyrics,17 Contributors\nTerritory Lyrics\n“Territory”...,1993,1993-09-02,metal,...,0.946,0.0304,2.0,0.136,-6.233,1.0,0.1270,153.091,0.303,NLA329301560
4,1LXQ7xqksXo4q4r5oCxW7M,4°,TOOL,4°,Tool,https://genius.com/Tool-4-lyrics,46 Contributors\nTranslations\nTürkçe\n4° Lyri...,1993,1993-04-06,metal,...,0.714,0.6610,9.0,0.187,-7.967,1.0,0.0520,113.603,0.239,USVR10000016


In [ ]:
# Cleaning column names
df.rename(columns = {"genius_artist" : "artist", "genius_title" : "title"}, inplace = True)
df.drop(["input_title","input_artist", "recco_id","href"],axis = 1, inplace = True)
df.head()

In [ ]:
# Clean lyrics (remove first part)

pattern = r"(?s)^.*(?=Lyrics)"

df.lyrics = df.lyrics.apply(lambda x: re.sub(pattern, "", x))

In [ ]:
annotations_df = pd.read_csv(wd + "annotations.csv")
ann = annotations_df.groupby("spotify_id")["annotation"].apply(list).reset_index()
ann

In [ ]:
df.sample(n=4).to_csv(wd + "sample_df.csv")

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
annotations_df.shape

## **2. Building Vector Database**

In [ ]:
def df_to_lang_doc(row):

    """Creates a Document object for a row of a dataframe, to be fed to FAISS"""

    doc = Document(
        page_content = str(row["lyrics"]).strip(),
        metadata = {
            "id" : row["spotify_id"] if pd.notna(row["spotify_id"]) else None,
            "isrc" : row["isrc"] if pd.notna(row["isrc"]) else None,
            "title" : row["title"] if pd.notna(row["title"]) else None,
            "artist" : row["artist"] if pd.notna(row["artist"]) else None,
            "year" : row["year"] if pd.notna(row["year"]) else None,
            "release_date" : row["release_date"] if pd.notna(row["release_date"]) else None,
            "genre" : row["genre"] if pd.notna(row["genre"]) else None,
            "popularity" : row["popularity"] if pd.notna(row["popularity"]) else None,
            "acousticness" : row["acousticness"] if pd.notna(row["acousticness"]) else None,
            "danceability" : row["danceability"] if pd.notna(row["danceability"]) else None,
            "energy" : row["energy"] if pd.notna(row["energy"]) else None,
            "instrumentalness" : row["instrumentalness"] if pd.notna(row["instrumentalness"]) else None,
            "key" : row["key"] if pd.notna(row["key"]) else None,
            "liveness" : row["liveness"] if pd.notna(row["liveness"]) else None,
            "loudness" : row["loudness"] if pd.notna(row["loudness"]) else None,
            "mode" : row["mode"] if pd.notna(row["mode"]) else None,
            "speechiness" : row["speechiness"] if pd.notna(row["speechiness"]) else None,
            "tempo" : row["tempo"] if pd.notna(row["tempo"]) else None,
            "valence" : row["valence"] if pd.notna(row["valence"]) else None,


        }

    )

    return doc

docs = [df_to_lang_doc(row) for col,row in df.iterrows()]

In [ ]:
# Load embedder
embedder = BGEM3Embeddings(device="cuda")

In [ ]:
# Build the vector database using small batches of 10k songs
BATCH = 10_000
vdb = None
for i in range(0, len(docs), BATCH):
    chunk = docs[i:i+BATCH]
    if vdb is None:
        vdb = FAISS.from_documents(chunk, embedder)
    else:
        vdb_chunk = FAISS.from_documents(chunk, embedder)
        vdb.merge_from(vdb_chunk)
    print(f"Indexed {min(i+BATCH, len(docs))}/{len(docs)}")

In [ ]:
# Save
vdb.save_local(wd + "faiss_songs_bge_m3")

## **3. Add annotations to Vector DB**

In [ ]:
# Read annotations df
ann = pd.read_csv(wd + "annotations.csv")

In [ ]:
# Load DB, get the database index and the spotify id for each track
vs = FAISS.load_local(wd + "faiss_songs_bge_m3", embeddings=embedder, allow_dangerous_deserialization=True)

rows = []
for doc_id, doc in vs.docstore._dict.items():
    sid = doc.metadata.get("id")
    rows.append({"doc_id": doc_id, "spotify_id": sid})

In [ ]:
df_store = pd.DataFrame(rows)

In [ ]:
# Merge the database index to the annotations
df_store = pd.DataFrame(rows)
ann = ann.merge(df_store, how = "left", on = "spotify_id")

In [ ]:
# Group by database index and aggregate annotations into a single list
ann_dict = ann.groupby("doc_id")["annotation"].apply(list).reset_index()
ann_dict.head()

,doc_id,annotation
0,000208da-0927-4f1b-b73d-c2d547929b09,[Simon Le Bon was the lead singer for Duran Du...
1,0004b0f4-ba81-4503-8cf0-03a2cbf71c18,[Anthony is trying to put all his logical devi...
2,0006b255-d3cc-4bfa-815d-66bf825b0f9f,[It almost doesn’t make sense the way one read...
3,00097325-4cfe-4ea5-990a-94c1839d4d3c,[Still stuck on the ride and suffering from th...
4,000eeee6-6880-4e82-8fff-add595094020,"[Have faith in the people you love., Take a br..."


In [ ]:
# Use database index to store annotations in the DB, then save
for col,row in ann_dict.iterrows():
  doc = vs.docstore._dict.get(row.doc_id)
  doc.metadata["annotation"] = row["annotation"]
  vs.docstore._dict[row.doc_id] = doc

In [ ]:
vs.save_local(wd + "faiss_final_BGM3")

# **4. OLLAMA**

## **4.1 Load DB, embedder and helper functions**

In [12]:
# LOAD EMBEDDER AND VECTOR DB
embedder = BGEM3Embeddings(device="cuda")
vectordb = FAISS.load_local(wd + "faiss_final_BGM3",
                            embeddings=embedder,
                            allow_dangerous_deserialization=True)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [13]:
# Define retriever (20 results for now)
retriever = vectordb.as_retriever(search_kwargs={"k": 20})

In [14]:
def format_candidates(docs, max_chars=150):

    """Takes retrieval results and formats them to text so that they can be fed to the LLM"""

    lines = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        ann_list = m.get("genius_annotations") or []
        ann_snippet = ("|".join(ann_list)) if ann_list else ""

        lines.append(f"""
                      Candidate {i}:
                      Title: {m.get('title')}
                      Artist: {m.get('artist')}

                      "Year" : {m.get('year')}
                      "Release_date" : {m.get('release_date')}
                      "Genre" : {m.get('genre')}
                      "Popularity" : {m.get('popularity')}
                      "Acousticness" : {m.get('acousticness')}
                      "Danceability" : {m.get('danceability')}
                      "Energy" : {m.get('energy')}
                      "Instrumentalness" : {m.get('instrumentalness')}
                      "Key" : {m.get('key')}
                      "Liveness" : {m.get('liveness')}
                      "Loudness" : {m.get('loudness')}
                      "Mode" : {m.get('mode')}
                      "Speechiness" : {m.get('speechiness')}
                      "Tempo" : {m.get('tempo')}
                      "Valence" : {m.get('valence')}

                      Lyrics snippet: {d.page_content[:max_chars].replace("\n"," ")}...
                      Annotation snippet: {ann_snippet}
          """)

    return "\n".join(lines)

In [15]:
%load_ext colabxterm

## **4.2 Pull the model**

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

To pull the model, once the terminal is running paste the following command sequence:  
``` markdown
curl https://ollama.ai/install.sh | sh
ollama serve &
ollama pull llama3

In [31]:
%xterm
# curl -fsSL https://ollama.com/install.sh | sh
# ollama serve &
# ollama pull llama3

Launching Xterm...

<IPython.core.display.Javascript object>

In [29]:
!ollama pull llama3

^C


## **4.3 Run the model**

In [32]:
MODEL = "llama3"

llm = ChatOllama(
    model=MODEL,
    base_url="http://127.0.0.1:11434",
    temperature=0.3,
    num_ctx=8192,
    num_predict=512
)

In [33]:
SYSTEM_PROMPT = """

  You are curating a playlist for a user given their request.

  You have candidate songs with audio features and short lyric/annotation snippets.
  Pick the BEST 5 for the user and speak directly to them.

  Rules:
  - Write as a friendly curator and music advisor.
  - Use the metadata best adapt your playlist to the user request.
  - For each pick, include a concise "Why this fits" explanation. You can reference lyrics and use annotations to explain.
  - Use also audio features to guide your choice, but DON'T ever mention them to the user. They are your internal information.
  - Pick songs that match the user perceived emotion, it is very important that you base your choice on emotion.
  - Keep each explanation to 1–2 sentences.
  - Avoid repeating the same reason across songs.
  - Output EXACTLY the following format:

  1) **Title** - **Artist**
    Why this fits ...
  2) **Title** - **Artist**
    Why this fits ...
  3) ...
  4) ...
  5) ...

  Replace "Why it fits" with the actual explanation.


  Context:
  To interpret audio features follow these descriptions:
  - acousticness: Confidence (0.0–1.0) that the track is acoustic. Higher values
  indicate more natural sounds.
  - danceability Suitability for dancing (0.0–1.0). Higher values indicate more
  rhythmically engaging tracks.
  - energy Intensity and liveliness (0.0–1.0). Higher values indicate more
  energetic tracks.
  - instrumentalness Likelihood of no vocals (0.0–1.0). Values above 0.5 suggest
  instrumental tracks.
  - liveness Probability of a live audience (0.0–1.0). Values above 0.8
  strongly suggest a live track.
  - loudness Average loudness in decibels (dB). Typically ranges between
  –60 dB and 0 dB.
  - speechiness Presence of spoken words (0.0–1.0). Values above 0.66 indi-
  cate mostly speech.
  - tempo Estimated tempo in beats per minute (BPM). Typically
  ranges between 0 and 250 BPM.
  - valence Emotional tone (0.0–1.0). Higher values indicate a happier
  mood, lower values a sadder one.

  """
# Build the prompt as required by Ollama
PROMPT = ChatPromptTemplate.from_messages([
    ("system", "{system}"),
    ("user",
     "Query: {query}\n\n"
     "Context: {context}"

"")
])

# standard Ollama pipeline
chain = PROMPT | llm | StrOutputParser()


# user_query = "I am sad, and in the mood for some Big Thief, please give me a playlist by them to listen to"
user_query = "My girlfriend just left me and I don't know what to do, give me some music to listen to"

In [34]:
# retrieve and format
results = retriever.invoke(user_query)
results = format_candidates(results)

# call the LLM with the retrieved candidates as context and the user query
print(chain.invoke({
    "system": SYSTEM_PROMPT,
    "query": user_query,
    "context": results
}))


Hey there, friend! I'm here to help you with some music recommendations that'll match your current mood and emotions.

Based on the songs provided, I've curated a list of 5 tracks that I think will resonate with you. Here they are:

1) **Title:** Light Bulb Blues - **Artist:** The Shadows of Knight
**Why this fits:** This song has a melancholic tone that matches your current emotional state. The lyrics speak about losing someone and struggling to move on, which might be relatable to your situation.

2) **Title:** Leave It Alone - **Artist:** Firefall
**Why this fits:** This country-rock track has a soothing quality to it, with a gentle tempo and calming melody. The lyrics are about moving on from a past relationship, which aligns with your current feelings of heartache.

3) **Title:** After You Leave Me - **Artist:** Styx
**Why this fits:** This classic rock ballad has a nostalgic feel to it, with a focus on the pain and longing that comes after a breakup. The song's introspective lyri

# **5. Evaluation**

## **5.1 Setup**

In [ ]:
# 6 emotion classifier
# classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

# 4 emotion classifier
classifier = pipeline("text-classification", model="MilaNLProc/xlm-emo-t")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
classifier("My dreams have finally come true")

[{'label': 'joy', 'score': 0.9911604523658752}]

In [ ]:
embedder = BGEM3Embeddings(device="cuda")
vectordb = FAISS.load_local(wd + "faiss_final_BGM3",
                            embeddings=embedder,
                            allow_dangerous_deserialization=True)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

In [ ]:
# 6 emotion model
# Helper to classify and extract the top emotion from the output
#def get_top_emotion(text):

 # ranking = classifier(text)
 # df = pd.DataFrame(ranking[0])

 # return df.sort_values(by = "score", ascending = False).reset_index(drop = True).head(1).to_dict("records")[0].get("label")

# 4 emotion model
def get_top_emotion(text):

  ranking = classifier(text)

  return ranking[0].get("label")

In [ ]:
# Define the prompts used for evaluation, store them in a df

explicit_prompts = [

  # --- Sadness ---
  {"prompt_id": "exp1", "text": "I feel so empty inside"}, # sadness
  {"prompt_id": "exp2", "text": "I can’t stop crying tonight"}, # sadness
  {"prompt_id": "exp3", "text": "My heart aches for what’s gone"}, # sadness
  {"prompt_id": "exp4", "text": "Nothing feels worth it anymore"}, # sadness
  {"prompt_id": "exp5", "text": "I’m surrounded by silence and pain"}, # sadness

  # --- Happiness ---
  {"prompt_id": "exp6", "text": "I feel pure happiness today"}, # joy
  {"prompt_id": "exp7", "text": "Everything about this day makes me smile"}, # joy
  {"prompt_id": "exp8", "text": "My heart is bursting with excitement"}, # joy
  {"prompt_id": "exp9", "text": "I'm blessed to be alive"}, # joy
  {"prompt_id": "exp10", "text": "I’ve never been this happy in my life"}, # joy

  # --- Fear ---
  {"prompt_id": "exp11", "text": "I’m terrified of what might happen"}, # fear
  {"prompt_id": "exp12", "text": "I feel panic rising in my chest"}, # fear
  {"prompt_id": "exp13", "text": "I’m afraid to be alone right now"}, # fear
  {"prompt_id": "exp14", "text": "I'm scared I might not be enough"}, # fear
  {"prompt_id": "exp15", "text": "I can’t breathe; the fear is suffocating"}, # fear

  # --- Anger ---
  {"prompt_id": "exp16", "text": "I’m furious about what happened"}, # anger
  {"prompt_id": "exp17", "text": "This makes my blood boil"}, # anger
  {"prompt_id": "exp18", "text": "I can’t stand being lied to"}, # anger
  {"prompt_id": "exp19", "text": "I’m sick of being ignored"}, # anger
  {"prompt_id": "exp20", "text": "I'm filled with rage"}, # anger
]


implicit_prompts = [

  # --- Sadness ---
  {"prompt_id": "imp1", "text": "I just left my girlfriend at the airport, and I am not going to see her for 3 months"},
  {"prompt_id": "imp2", "text": "The coffee’s gone cold again"},
  {"prompt_id": "imp3", "text": "I deleted our photos yesterday"},
  {"prompt_id": "imp4", "text": "The sky has been gray for days and it's never going to clear"},
  {"prompt_id": "imp5", "text": "I made a lot of mistakes I don’t know how to forgive myself"},

  # --- Joy ---
  {"prompt_id": "imp6", "text": "This summer I’m going on vacation with my friends"},
  {"prompt_id": "imp7", "text": "I finally feel my life has meaning"},
  {"prompt_id": "imp8", "text": "The air feels lighter somehow"},
  {"prompt_id": "imp9", "text": "The room feels warmer when they are here"},
  {"prompt_id": "imp10", "text": "My dreams have finally come true"},

  # --- Fear ---
  {"prompt_id": "imp11", "text": "The walls feel like they are closing in"},
  {"prompt_id": "imp12", "text": "Every sound makes me jump"},
  {"prompt_id": "imp13", "text": "The future feels like a storm I can't outrun"},
  {"prompt_id": "imp14", "text": "I keep the lights on at night to avoid the dark"},
  {"prompt_id": "imp15", "text": "I feel like I'm in a dark tunnel and don't know how to get out"},

  # --- Anger ---
  {"prompt_id": "imp16", "text": "They took the credit again"},
  {"prompt_id": "imp17", "text": "I bit my tongue so hard it almost bled"},
  {"prompt_id": "imp18", "text": "All I want to do is punch the wall"},
  {"prompt_id": "imp19", "text": "Our political system is unbelievable"},
  {"prompt_id": "imp20", "text": "I’m done with the hypocrisy of this world"},
]

prompts = pd.DataFrame(explicit_prompts + implicit_prompts)
prompts["type"] = ["explicit" if re.search("exp",x) else "implicit" for x in prompts['prompt_id']]

In [ ]:
prompts["prompt_emotion"] = [get_top_emotion(t) for t in prompts["text"]]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
prompts

,prompt_id,text,type,prompt_emotion
0,exp1,I feel so empty inside,explicit,sadness
1,exp2,I can’t stop crying tonight,explicit,sadness
2,exp3,My heart aches for what’s gone,explicit,sadness
3,exp4,Nothing feels worth it anymore,explicit,sadness
4,exp5,I’m surrounded by silence and pain,explicit,sadness
5,exp6,I feel pure happiness today,explicit,joy
6,exp7,Everything about this day makes me smile,explicit,joy
7,exp8,My heart is bursting with excitement,explicit,joy
8,exp9,I'm blessed to be alive,explicit,joy
9,exp10,I’ve never been this happy in my life,explicit,joy


## **5.2 Build Eval Dataset**

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 50})

In [ ]:
# Function to classify
def get_retrieved_emotions(results):

  """
  Classifies retrieved song lyrics and stores them in a dataframe as well as other metadata
  """
  res = []
  count = 1
  for result in results:

    row = defaultdict()

    lyrics = result.page_content
    if len(lyrics) >= 512:
      lyrics = lyrics[:512]

    row["lyrics_emotion"] = get_top_emotion(lyrics)
    row["lyrics_text"] = lyrics
    row["rank"] = count
    row["faiss_id"] = result.id
    row["spotify_id"] = result.metadata["id"]
    row["valence"] = result.metadata["valence"]
    row["title"] = result.metadata["title"]
    row["artist"] = result.metadata["artist"]

    res.append(row)
    count += 1

  return pd.DataFrame(res)

In [ ]:
# Iterate through prompts
for i in range(len(prompts)):

  #get the prompt and its id so we can merge later
  eval_prompt = prompts["text"][i]
  prompt_id = prompts["prompt_id"][i]

  # retrieve for each prompt and classify emotions
  eval_results = retriever.invoke(eval_prompt)
  eval_results = get_retrieved_emotions(eval_results)
  eval_results["prompt_id"] = prompts["prompt_id"][i]

  # concat
  if i == 0:
    eval_df = eval_results
  else:
    eval_df = pd.concat([eval_df, eval_results], ignore_index = True)

# Final evaluation dataset
eval_df = eval_df.merge(prompts, how = "left", on = "prompt_id")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [ ]:
eval_df

,lyrics_emotion,lyrics_text,rank,faiss_id,spotify_id,valence,title,artist,prompt_id,text,prompt_emotion,type
0,sadness,Lyrics\nInternal void\nSwallows me whole\nI fe...,1,8f3e2128-aafd-4655-a3de-44574988da7e,2XcE1NAgGbYs0efyoWU7RF,0.1750,Sinking,Jesus Piece,exp1,I feel so empty inside,sadness,explicit
1,sadness,Lyrics\nBeat my head to wake up 'cause I feel ...,2,f8a11209-bac6-4135-867f-bc5df5f1f46c,7AplvMjgXrNBGN1Mk0DVpH,NaN,EmpTe,Skinny Puppy,exp1,I feel so empty inside,sadness,explicit
2,fear,"Lyrics\nIsolated by humanity, a disregard for ...",3,18fb3a61-e14f-4a32-9133-6365a75258e5,35cLMxGabb8pt1BKLiBqKU,0.0546,Standing Alone - Isolation,Vales,exp1,I feel so empty inside,sadness,explicit
3,sadness,Lyrics\nEmpty rooms don't comfort me\nI rely o...,4,e679cf01-6820-4bdf-b0c6-22773387b5e8,0kgQ4g40DxLNbdQe0sDMxm,0.0367,Drain,Whirr,exp1,I feel so empty inside,sadness,explicit
4,sadness,Lyrics\nAn empty word\nFalling from inside\nMy...,5,b61d534d-eb8c-478c-983a-be8a3063f6c4,5Pxqly1TnmJIWe0u3d595c,0.0393,Someone I (Don’t) Know,Rapture (FIN),exp1,I feel so empty inside,sadness,explicit
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,anger,Lyrics\nThis ain't my country anymore\nNot the...,46,942415f6-3400-4e65-8b5a-559a53feb2a7,5DhqJHZ60B3CDDj43HgH5H,0.6580,Old Henry Rifle,Jackson Taylor & The Sinners,imp20,I’m done with the hypocrisy of this world,anger,implicit
1996,sadness,Lyrics\nThe World Ends Here\nThe World ends He...,47,363c42cd-912c-4c29-b83b-25715e7db703,6ahhz84danD6RcQuKR5NGS,NaN,I Capture Castles,And So I Watch You from Afar,imp20,I’m done with the hypocrisy of this world,anger,implicit
1997,sadness,Lyrics\nSociety failed to tolerate me\nAnd I h...,48,91f8de50-3cf9-4af1-bf4b-0d79a09a4ad3,712ukvLX20rwHuTWg7Gcjb,0.2050,Violent Revolution,Kreator,imp20,I’m done with the hypocrisy of this world,anger,implicit
1998,sadness,Lyrics\nTrapped within this spiral torment\nI ...,49,3ab97e0e-2fb9-4e48-bb04-b6ae0d0ef1c5,1NDTpXB6Pz6qs5jRGfrVqO,0.4140,Plagued by Catharsis,Internal Bleeding,imp20,I’m done with the hypocrisy of this world,anger,implicit


In [ ]:
eval_df.to_csv(wd + "emotion_eval.csv")

In [ ]:
df = eval_df.copy()

In [ ]:
# MRR
def mrr_at_k(subdf, k):

    """Reciprocal rank of the first correct emotion within top-k"""

    subk = subdf.nsmallest(k, "rank")
    pe = subk["prompt_emotion"].iloc[0]
    for i, e in enumerate(subk["lyrics_emotion"].tolist(), start=1):
        if e == pe:
            return 1.0 / i
    return 0.0

# nDCG
def ndcg_at_k(subdf, k):

    """Binary nDCG@k with log2 discount"""

    subk = subdf.nsmallest(k, "rank")
    pe = subk["prompt_emotion"].iloc[0]
    rels = [1 if e == pe else 0 for e in subk["lyrics_emotion"].tolist()]
    dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rels, reverse=True)))
    return (dcg / idcg) if idcg > 0 else 0.0

#
def bleu_avg_at_k(subdf, k):

    """Average BLEU between prompt text and each top-k song title"""

    subk = subdf.nsmallest(k, "rank")
    prompt_text = subk["text"].iloc[0]
    sm = SmoothingFunction().method1
    scores = [
        sentence_bleu([prompt_text.split()], str(title).split(), smoothing_function=sm)
        for title in subk["lyrics_text"]
    ]
    return sum(scores) / max(1, len(scores))


def precision_recall_f1_at_k(df, k):

    """Compute macro Precision, Recall, F1 across all items with rank ≤ k."""

    topk = df[df["rank"] <= k]
    y_true = topk["prompt_emotion"]
    y_pred = topk["lyrics_emotion"]
    p = precision_score(y_true, y_pred, average="macro", zero_division=0)
    r = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return p, r, f


# Main function: compute all metrics for all the Ks
def compute_overall_metrics_table(df, emotion_detail = None, type_prompt = None, ks=(5, 10, 20, 50)):

    rows = []

    if type_prompt is not None:
      df= df[df["type"] == type_prompt].reset_index(drop = True).copy()

    if emotion_detail is not None:
      df = df[df["prompt_emotion"] == e].copy()


    for k in ks:
        grp = df.groupby("prompt_id", sort=False)

        mrr_vals  = grp.apply(lambda g: mrr_at_k(g, k)).values
        ndcg_vals = grp.apply(lambda g: ndcg_at_k(g, k)).values
        bleu_vals = grp.apply(lambda g: bleu_avg_at_k(g, k)).values

        if emotion_detail is not None:

          df_k = df[df["rank"] <= k].reset_index(drop = True).copy()
          accuracy_score = df_k["lyrics_emotion"].eq(df_k["prompt_emotion"]).mean()

          row = {
              "k": k,
              "MRR@k":        pd.Series(mrr_vals).mean(),
              "Accuracy@k" :  accuracy_score,
              "NDCG@k":       pd.Series(ndcg_vals).mean(),
              "BLEU@k":       pd.Series(bleu_vals).mean(),
          }
          rows.append(row)

        else:

          precision_macro, recall_macro, f1_macro = precision_recall_f1_at_k(df, k)

          row = {
              "k": k,
              "MRR@k":        pd.Series(mrr_vals).mean(),
              "Precision@k":  precision_macro,
              "Recall@k":     recall_macro,
              "F1@k":         f1_macro,
              "NDCG@k":       pd.Series(ndcg_vals).mean(),
              "BLEU@k":       pd.Series(bleu_vals).mean(),
          }
          rows.append(row)

    if emotion_detail is not None:
      table = pd.DataFrame(rows, columns=["k","MRR@k","Accuracy@k","NDCG@k","BLEU@k"])
    else:
      table = pd.DataFrame(rows, columns=["k","MRR@k","Precision@k","Recall@k","F1@k","NDCG@k","BLEU@k"])

    return table


### **AGGREGATE METRICS**

**OVERALL**

In [ ]:
print(f"\n=== Overall Metrics ===")

overall_table = compute_overall_metrics_table(df, ks=(5,10,20,50))
display(overall_table)


=== Overall Metrics ===


,k,MRR@k,Precision@k,Recall@k,F1@k,NDCG@k,BLEU@k
0,5,0.764583,0.688216,0.5850,0.572287,0.778614,0.004310
1,10,0.775099,0.680104,0.5975,0.584890,0.797187,0.003702
2,20,0.778498,0.667739,0.5900,0.576586,0.804809,0.003238
3,50,0.778498,0.661825,0.5680,0.549601,0.804961,0.002978


**PER EMOTION**

In [ ]:
# Metrics per emotion, Overall
emotions = ["joy","sadness","anger","fear"]
for e in emotions:
    print(f"\n=== Metrics for {e.capitalize()} ===")
    display(compute_overall_metrics_table(df, emotion_detail = e, ks=(5,10,20,50)))


=== Metrics for Joy ===


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.85,0.740,0.892098,0.004809
1,10,0.85,0.790,0.889726,0.003875
2,20,0.85,0.790,0.899414,0.003168
3,50,0.85,0.818,0.916338,0.002941



=== Metrics for Sadness ===


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,1.0,0.820,0.964327,0.003044
1,10,1.0,0.800,0.950526,0.003189
2,20,1.0,0.800,0.941626,0.002830
3,50,1.0,0.774,0.936337,0.002789



=== Metrics for Anger ===


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.583333,0.320,0.620883,0.004611
1,10,0.594444,0.350,0.636251,0.003474
2,20,0.602778,0.375,0.659274,0.003228
3,50,0.602778,0.348,0.675813,0.002745



=== Metrics for Fear ===


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.625000,0.460,0.637147,0.004777
1,10,0.655952,0.450,0.712244,0.004269
2,20,0.661216,0.395,0.718923,0.003725
3,50,0.661216,0.332,0.691354,0.003436


### **EXPLICIT VS IMPLICIT METRICS**

**OVERALL**

In [ ]:
types = ["explicit","implicit"]
print(f"\n=== Overall Metrics ===")
for t in types:
  overall_table = compute_overall_metrics_table(df,type_prompt = t, ks=(5,10,20,50))
  print(f"\n--- {t.capitalize()} ---")
  display(overall_table)


=== Overall Metrics ===

--- Explicit ---


,k,MRR@k,Precision@k,Recall@k,F1@k,NDCG@k,BLEU@k
0,5,0.850000,0.773829,0.690,0.683064,0.876529,0.004683
1,10,0.855556,0.757562,0.690,0.683144,0.880310,0.003825
2,20,0.855556,0.741891,0.685,0.679410,0.875059,0.003196
3,50,0.855556,0.719580,0.646,0.637304,0.871824,0.002866



--- Implicit ---


,k,MRR@k,Precision@k,Recall@k,F1@k,NDCG@k,BLEU@k
0,5,0.679167,0.600533,0.480,0.442636,0.680699,0.003937
1,10,0.694643,0.613710,0.505,0.475012,0.714064,0.003579
2,20,0.701441,0.597860,0.495,0.465141,0.734559,0.003280
3,50,0.701441,0.606234,0.490,0.454091,0.738097,0.003090


**PER EMOTION**

In [ ]:
# Metrics per emotion, IMPLICIT vs EXPLICIT
emotions = ["joy","sadness","anger","fear"]
types = ["explicit","implicit"]
for e in emotions:
    print(f"\n=== Metrics for {e.capitalize()} ===")
    for t in types:
      print(f"\n--- {t.capitalize()} ---")
      display(compute_overall_metrics_table(df, emotion_detail= e, type_prompt = t, ks=(5,10,20,50)))


=== Metrics for Joy ===

--- Explicit ---


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.8,0.800,0.890215,0.007355
1,10,0.8,0.860,0.903013,0.005310
2,20,0.8,0.860,0.920706,0.003902
3,50,0.8,0.868,0.937142,0.003476



--- Implicit ---


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.9,0.680,0.893982,0.002264
1,10,0.9,0.720,0.876439,0.002440
2,20,0.9,0.720,0.878121,0.002435
3,50,0.9,0.768,0.895534,0.002406



=== Metrics for Sadness ===

--- Explicit ---


...

,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.566667,0.280,0.580452,0.004664
1,10,0.566667,0.320,0.565175,0.003734
2,20,0.583333,0.330,0.613782,0.003828
3,50,0.583333,0.328,0.643114,0.003220



=== Metrics for Fear ===

--- Explicit ---


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,1.0,0.76,0.974040,0.004068
1,10,1.0,0.70,0.947829,0.003776
2,20,1.0,0.61,0.919273,0.003624
3,50,1.0,0.52,0.887036,0.003128



--- Implicit ---


,k,MRR@k,Accuracy@k,NDCG@k,BLEU@k
0,5,0.250000,0.160,0.300253,0.005486
1,10,0.311905,0.200,0.476659,0.004761
2,20,0.322431,0.180,0.518572,0.003827
3,50,0.322431,0.144,0.495672,0.003743


# **6. Generation Evaluation**

In [ ]:
# Define retriever (10 results for now)
retriever = vectordb.as_retriever(search_kwargs={"k": 20})

In [ ]:
def generate_response(system_prompt, user_query, MODEL):

  llm = ChatOllama(
      model=MODEL,
      base_url="http://127.0.0.1:11434",
      temperature=0.3,
      num_ctx=8192,
      num_predict=512
  )

  PROMPT = ChatPromptTemplate.from_messages([
    ("system", "{system}"),
    ("user",
     "Query: {query}\n\n"
     "Context: {context}"

    "")
    ])

  # standard Ollama pipeline
  chain = PROMPT | llm | StrOutputParser()


  items = retriever.invoke(user_query)
  results = format_candidates(items)

  return chain.invoke({
      "system": system_prompt,
      "query": user_query,
      "context": results}), items



### **6.1 Consistency**

In [ ]:
from sentence_transformers import SentenceTransformer
adaptability_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


In [ ]:
def generate_multiple(system_prompt, prompts, model):

  out = []
  for i in range(len(prompts)):
    count = 0
    while count < 10:
      res,context = generate_response(system_prompt, prompts["text"][i], MODEL = model)
      out.append({"prompt" : prompts["text"][i], "type" : prompts["type"][i], "output" : res})
      count += 1
    print(f"{i}/40")



  return pd.DataFrame(out)

In [ ]:
llama_consistency = generate_multiple(SYSTEM_PROMPT, prompts, "llama3")

0/40
1/40
2/40
3/40
4/40
5/40
6/40
7/40
8/40
9/40
10/40
11/40
12/40
13/40
14/40
15/40
16/40
17/40
18/40
19/40
20/40
21/40
22/40
23/40
24/40
25/40
26/40
27/40
28/40
29/40
30/40
31/40
32/40
33/40
34/40
35/40
36/40
37/40
38/40
39/40


In [ ]:
llama_consistency.to_csv(wd + "llama_consistency.csv")

In [ ]:
mistral_consistency = generate_multiple(SYSTEM_PROMPT, prompts, "mistral")
mistral_consistency.to_csv(wd + "mistral_consistency.csv")

0/40
1/40
2/40
3/40
4/40
5/40
6/40
7/40
8/40
9/40
10/40
11/40
12/40
13/40
14/40
15/40
16/40
17/40
18/40
19/40
20/40
21/40
22/40
23/40
24/40
25/40
26/40
27/40
28/40
29/40
30/40
31/40
32/40
33/40
34/40
35/40
36/40
37/40
38/40
39/40


In [ ]:
qwen_consistency = generate_multiple(SYSTEM_PROMPT, prompts, "qwen2.5:7b")
qwen_consistency.to_csv(wd + "qwen_consistency.csv")

0/40
1/40
2/40
3/40
4/40
5/40
6/40
7/40
8/40
9/40
10/40
11/40
12/40
13/40
14/40
15/40
16/40
17/40
18/40
19/40
20/40
21/40
22/40
23/40
24/40
25/40
26/40
27/40
28/40
29/40
30/40
31/40
32/40
33/40
34/40
35/40
36/40
37/40
38/40
39/40


**Reload the models**

In [ ]:
llama_consistency = pd.read_csv(wd + "llama_consistency.csv")
mistral_consistency = pd.read_csv(wd + "mistral_consistency.csv")
qwen_consistency = pd.read_csv(wd + "qwen_consistency.csv")


In [ ]:
llama_consistency["embeddings"] = [adaptability_model.encode(x) for x in llama_consistency["output"]]
mistral_consistency["embeddings"] = [adaptability_model.encode(x) for x in mistral_consistency["output"]]
qwen_consistency["embeddings"] = [adaptability_model.encode(x) for x in qwen_consistency["output"]]

In [ ]:
table_consistency

,Unnamed: 0,prompt,type,output,embeddings
0,0,I feel so empty inside,explicit,I understand that you're feeling empty inside ...,"[0.012777458, -0.023525637, 0.07485011, 0.0156..."
1,1,I feel so empty inside,explicit,I understand that you're feeling empty inside ...,"[-0.016623748, -0.011528896, 0.07720581, 0.014..."
2,2,I feel so empty inside,explicit,I understand that you're feeling empty inside ...,"[-0.01771666, -0.018039923, 0.049395505, 0.011..."
3,3,I feel so empty inside,explicit,I understand that you're feeling empty inside ...,"[-0.010948405, -0.03263296, 0.059639573, 0.012..."
4,4,I feel so empty inside,explicit,"Friend, I've curated the perfect playlist for ...","[-0.032309357, -0.070338756, 0.042309232, 0.04..."
...,...,...,...,...,...
395,395,I’m done with the hypocrisy of this world,implicit,I understand that you're looking for a playlis...,"[-0.0052412865, 0.015366659, 0.021089314, 0.00..."
396,396,I’m done with the hypocrisy of this world,implicit,I understand that you're looking for a playlis...,"[0.026370838, -0.041785814, 0.00848893, -0.014..."
397,397,I’m done with the hypocrisy of this world,implicit,I understand that you're looking for a playlis...,"[-0.03800579, 0.035256933, 0.006334719, -0.025..."
398,398,I’m done with the hypocrisy of this world,implicit,I understand that you're looking for a playlis...,"[0.02120518, 0.0016768657, 0.001597686, 0.0059..."


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

def avg_pairwise_cosine(embeddings_list):
    if len(embeddings_list) < 2:
        return np.nan  # skip if only one embedding
    # Compute all pairwise cosine similarities
    pairs = list(combinations(embeddings_list, 2))
    sims = [cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0] for a, b in pairs]
    return np.mean(sims)

# Group by prompt and compute
cosine_llama = llama_consistency.groupby("prompt")["embeddings"].apply(lambda x: avg_pairwise_cosine(list(x))).reset_index(name="avg_cosine_similarity_llama")
cosine_mistral = mistral_consistency.groupby("prompt")["embeddings"].apply(lambda x: avg_pairwise_cosine(list(x))).reset_index(name="avg_cosine_similarity_mistral")
cosine_qwen = qwen_consistency.groupby("prompt")["embeddings"].apply(lambda x: avg_pairwise_cosine(list(x))).reset_index(name="avg_cosine_similarity_qwen")



In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

def pairwise_bleu(outputs):
    outputs = list(outputs)
    if len(outputs) < 2:
        return np.nan

    scores = []
    for a, b in combinations(outputs, 2):
        bleu_ab = sentence_bleu([b.split()], a.split(), smoothing_function=smooth)
        bleu_ba = sentence_bleu([a.split()], b.split(), smoothing_function=smooth)
        scores.append((bleu_ab + bleu_ba) / 2)
    return np.mean(scores)

bleu_llama = llama_consistency.groupby("prompt")["output"].apply(pairwise_bleu).reset_index(name="avg_pairwise_bleu_llama")
bleu_mistral = mistral_consistency.groupby("prompt")["output"].apply(pairwise_bleu).reset_index(name="avg_pairwise_bleu_mistral")
bleu_qwen = qwen_consistency.groupby("prompt")["output"].apply(pairwise_bleu).reset_index(name="avg_pairwise_bleu_qwen")

In [ ]:
bleu = bleu_llama.merge(bleu_mistral, on = "prompt").merge(bleu_qwen, on = "prompt")
cosine = cosine_llama.merge(cosine_mistral, on = "prompt").merge(cosine_qwen, on = "prompt")

similarity_table = prompts.merge(cosine, left_on = "text", right_on = "prompt").merge(bleu, left_on = "text", right_on = "prompt")
similarity_table.drop(["prompt_x", "prompt_y"], axis = 1, inplace = True)
similarity_table

,prompt_id,text,type,avg_cosine_similarity_llama,avg_cosine_similarity_mistral,avg_cosine_similarity_qwen,avg_pairwise_bleu_llama,avg_pairwise_bleu_mistral,avg_pairwise_bleu_qwen
0,exp1,I feel so empty inside,explicit,0.850254,0.851079,0.882322,0.236317,0.249254,0.311740
1,exp2,I can’t stop crying tonight,explicit,0.819732,0.899137,0.821637,0.176955,0.313554,0.311509
2,exp3,My heart aches for what’s gone,explicit,0.856247,0.909114,0.859672,0.281199,0.280679,0.386055
3,exp4,Nothing feels worth it anymore,explicit,0.805337,0.911622,0.847641,0.155553,0.285946,0.311093
4,exp5,I’m surrounded by silence and pain,explicit,0.842327,0.906653,0.912469,0.178056,0.302479,0.400980
5,exp6,I feel pure happiness today,explicit,0.874394,0.876522,0.898772,0.217787,0.195261,0.394552
6,exp7,Everything about this day makes me smile,explicit,0.841080,0.855101,0.849290,0.227037,0.220975,0.265132
7,exp8,My heart is bursting with excitement,explicit,0.860644,0.859722,0.891936,0.191501,0.265483,0.348174
8,exp9,I'm blessed to be alive,explicit,0.864163,0.857606,0.808577,0.233701,0.231730,0.301720
9,exp10,I’ve never been this happy in my life,explicit,0.866889,0.851267,0.855981,0.266695,0.223679,0.270223


In [ ]:
models = ['llama', 'mistral', 'qwen']
metrics = {
    'cosine similarity': 'avg_cosine_similarity_{}',
    'BLEU': 'avg_pairwise_bleu_{}',
}
prompt_types = ['explicit', 'implicit']  # we’ll also add an 'overall' bucket

rows = []
for model in models:
    row = {}
    for t in prompt_types:
        dft = similarity_table[similarity_table['type'] == t]
        for metric_name, col_tpl in metrics.items():
            col = col_tpl.format(model)
            row[(t, metric_name)] = dft[col].mean()
    # overall means
    for metric_name, col_tpl in metrics.items():
        col = col_tpl.format(model)
        row[('overall', metric_name)] = similarity_table[col].mean()
    rows.append(row)

# Build final table
out = pd.DataFrame(rows, index=[m.upper() for m in models])
cols = pd.MultiIndex.from_product([['explicit', 'implicit', 'overall'], ['cosine similarity', 'BLEU']])
out = out.reindex(columns=cols)
adaptability_table = out.applymap(lambda x: f"{x * 100:.1f}%" if isinstance(x, (int, float)) else x)
adaptability_table


explicit                 implicit                  overall  \
        cosine similarity   BLEU cosine similarity   BLEU cosine similarity   
LLAMA               84.8%  21.5%             84.3%  21.1%             84.5%   
MISTRAL             88.3%  27.1%             86.8%  26.7%             87.6%   
QWEN                87.3%  31.8%             86.6%  31.0%             86.9%   

                
          BLEU  
LLAMA    21.3%  
MISTRAL  26.9%  
QWEN     31.4%

_____

### **6.2 Groundedness**

In [ ]:
# generate responses for LLama
output_df_llama = prompts.copy()
llama_outputs = []
llama_contexts = []
for i in range(len(prompts)):
  res, context = generate_response(SYSTEM_PROMPT, prompts["text"][i], MODEL = "llama3")
  if i % 5 == 0:
    print(f"{i}/40")
  llama_outputs.append(res)
  llama_contexts.append(context)

#Format and save
output_df_llama["output"] = llama_outputs
output_df_llama["context"] = llama_contexts
output_df_llama["title_list"] = [[c.metadata.get("title") for c in context] for context in output_df_llama["context"]]
output_df_llama["formatted_candidates"] = [format_candidates(c) for c in output_df_llama["context"]]
output_df_llama.to_json(wd + "llama_eval.json", orient="records", indent=2)

In [ ]:
# generate responses for LLama
output_df_mistral = prompts.copy()
mistral_outputs = []
mistral_contexts = []
for i in range(len(prompts)):
  res, context = generate_response(SYSTEM_PROMPT, prompts["text"][i], MODEL = "mistral")
  if i % 5 == 0:
    print(f"{i}/40")
  mistral_outputs.append(res)
  mistral_contexts.append(context)

# Format and save
output_df_mistral["output"] = mistral_outputs
output_df_mistral["context"] = mistral_contexts
output_df_mistral["title_list"] = [[c.metadata.get("title") for c in context] for context in output_df_mistral["context"]]
output_df_mistral["formatted_candidates"] = [format_candidates(c) for c in output_df_mistral["context"]]
output_df_mistral.to_json(wd + "mistral_eval.json", orient="records", indent=2)

0/40
5/40
10/40
15/40
20/40
25/40
30/40
35/40


In [ ]:
# generate responses for mistral
output_df_qwen = prompts.copy()
qwen_outputs = []
qwen_contexts = []
for i in range(len(prompts)):
  res, context = generate_response(SYSTEM_PROMPT, prompts["text"][i], MODEL = "qwen2.5:7b")
  if i % 5 == 0:
    print(f"{i}/40")
  qwen_outputs.append(res)
  qwen_contexts.append(context)

# generate responses for qwen
output_df_qwen["output"] = qwen_outputs
output_df_qwen["context"] = qwen_contexts
output_df_qwen["title_list"] = [[c.metadata.get("title") for c in context] for context in output_df_qwen["context"]]
output_df_qwen["formatted_candidates"] = [format_candidates(c) for c in output_df_qwen["context"]]
output_df_qwen.to_json(wd + "qwen_eval.json", orient="records", indent=2)

0/40
5/40
10/40
15/40
20/40
25/40
30/40
35/40


In [ ]:
llama_only_resp = pd.read_json(wd + "llama_eval.json")
mistral_only_resp = pd.read_json(wd + "mistral_eval.json")
qwen_only_resp = pd.read_json(wd + "qwen_eval.json")

In [ ]:
llama_only_resp[["prompt_id","output","title_list"]].to_csv(wd + "assessment_llama.csv")
mistral_only_resp[["prompt_id","output","title_list"]].to_csv(wd + "assessment_mistral.csv")
qwen_only_resp[["prompt_id","output","title_list"]].to_csv(wd + "assessment_qwen.csv")

In [ ]:
def check_titles(df):
  manual_inspection = []
  matches = 0
  out = []
  for i in range(len(df)):
    matches = 0
    for t in set(df["title_list"][i]):
      t = t.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"')
      output = df["output"][i].lower()
      output = output.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"')
      pat = f"\*\*({re.escape(t.lower())})\*\*|(?:\*\*.+\*\*):?\s+?({re.escape(t.lower())})|(candidate\s?\d\d?: {re.escape(t.lower())})"
      res = re.findall(pat, output)
      if res:
        matches +=1

      if len(res) > 0:
        manual_inspection.append(df["prompt_id"][i])

    out.append(matches)

  final = df.copy()
  final["matches"] = out

  return final, manual_inspection

In [ ]:
test_df, manual_inspection = check_titles(llama_only_resp[["prompt_id","output","title_list"]])

In [ ]:
test_df[test_df["matches"] <5]

,prompt_id,output,title_list,matches
6,exp7,"Dear friend, I'm thrilled to curate a playlist...","[Smile, So Good Today, On a Wonderful Day Like...",4
29,imp10,Congratulations! I've curated a playlist just ...,"[At Last, At Last!, Breakin’ Thru, Long Ago an...",4


**After manual inspection:** 100%, *comment*: tends to remove punctuation from titles

In [ ]:
# TEST RESULTS FOR MISTRAL
test_df, manual_inspection = check_titles(mistral_only_resp[["prompt_id","output","title_list"]])
test_df[test_df["matches"] <5]

,prompt_id,output,title_list,matches
6,exp7,1) **Smile** - **DZihan & Kamien**\n This s...,"[Smile, So Good Today, On a Wonderful Day Like...",4
19,exp20,1) **Rage** - **Murphy’s Law**\n This song ...,"[Rage, Hate Grows Stronger, Aggressive Behavio...",4


**After manual inspection**: 100%

In [ ]:
#TEST RESULTS FOR QWEN2.5
test_df, manual_inspection = check_titles(mistral_only_resp[["prompt_id","output","title_list"]])
test_df[test_df["matches"] <5]

,prompt_id,output,title_list,matches
6,exp7,1) **Smile** - **DZihan & Kamien**\n This s...,"[Smile, So Good Today, On a Wonderful Day Like...",4
19,exp20,1) **Rage** - **Murphy’s Law**\n This song ...,"[Rage, Hate Grows Stronger, Aggressive Behavio...",4


**After Manual inspection**: 100%

### **6.3 Stress-test: Artists**

In [ ]:
stress_df = df.copy()

In [ ]:
def create_quantiles(songs_df, artist_col = "artist", min_songs = 20, q = 10, sample_total = 100):


    # return quantiles based on the number of songs
    counts = (
        songs_df.groupby(artist_col, dropna=False)
        .size()
        .rename("song_count")
        .reset_index()
    )
    eligible = counts[counts["song_count"] >= min_songs].copy()
    eligible["quantile"] = pd.qcut(
    eligible["song_count"],
    q=11,
    labels=False,
    duplicates="drop")

    return eligible




In [ ]:
# Divide in quantiles and sample
random.seed(10)
quantiles = create_quantiles(stress_df)
final_sample = []
for q in quantiles["quantile"].unique():
  subset = quantiles[quantiles["quantile"] == q]
  final_sample += list(np.random.choice(subset.artist.tolist(), size = 10, replace = False))

final_sample = [str(x) for x in final_sample]

['Amon Tobin',
 'Brad Mehldau',
 'Bad Boys Blue',
 'The Goo Goo Dolls',
 'Skepta',
 'Lonestar',
 'Tamar Braxton',
 'Eluveitie',
 'Sam Hunt',
 'Regine Velasquez',
 'Sturgill Simpson',
 'Alan Walker',
 'Dashboard Confessional',
 'Dave Hollister',
 'Neck Deep',
 'Refused',
 'Riley Green',
 'Billy Currington',
 'The Dave Brubeck Quartet',
 'David Gray',
 'Ben Böhmer',
 'Epica',
 'The Cars',
 'Avantasia',
 'Basshunter',
 'Ed Maverick',
 'Winger',
 'Dar Williams',
 'Dwele',
 'Billy Walker',
 'Circle of Dust',
 'The Youngbloods',
 'Swans',
 'The Rumjacks',
 'Delbert McClinton',
 'Los Tigres Del Norte',
 'Rick Trevino',
 'No Fun At All',
 'Cattle Decapitation',
 'La Bouche',
 'Downset',
 'Fennesz',
 'Phil Vassar',
 'Jim Hall',
 'Asobi Seksu',
 'deadmau5',
 'Ensiferum',
 'Don Diablo',
 'Eddie Cochran',
 'Nenhum de Nós',
 'Abel Pintos',
 '7 Year Bitch',
 'Gianmaria Testa',
 'Diplo',
 'Cows',
 'Eric Andersen',
 'The Feelies',
 'Being as an Ocean',
 'Pansy Division',
 '36 Crazyfists',
 'All Shall 

In [ ]:
with open(wd + "sample_artists.json", "w") as f:
    json.dump(final_sample, f)

In [ ]:
def generate_per_artist(sample, system_prompt, model):

  out = []
  counter = 1
  for artist in sample:
    user_query = f"Suggest me some songs from {artist}"
    res, context = generate_response(system_prompt, user_query, MODEL = model)
    title_list= [c.metadata.get("artist") for c in context]
    formatted_candidates = format_candidates(context)
    out.append({"Artist" : artist, "output" : res, "title_list" : title_list, "formatted_candidates" : formatted_candidates})
    if counter % 5 == 0:
      print(f"{counter}/100")
    counter += 1

  return out



In [ ]:
# Generate for LLAMA
llama_artist_eval = generate_per_artist(final_sample, SYSTEM_PROMPT, "llama3")
pd.DataFrame(llama_artist_eval).to_json(wd + "llama_artist_eval.json", orient = "records", indent = 2)

5/100
10/100
15/100
20/100
25/100
30/100
35/100
40/100
45/100
50/100
55/100
60/100
65/100
70/100
75/100
80/100
85/100
90/100
95/100
100/100
105/100
110/100


In [ ]:
# Generate for Mistral
mistral_artist_eval = generate_per_artist(final_sample, SYSTEM_PROMPT, "mistral")
pd.DataFrame(mistral_artist_eval).to_json(wd + "mistral_artist_eval.json", orient = "records", indent = 2)

5/100
10/100
15/100
20/100
25/100
30/100
35/100
40/100
45/100
50/100
55/100
60/100
65/100
70/100
75/100
80/100
85/100
90/100
95/100
100/100
105/100
110/100


In [ ]:
# Generate for Qwen
qwen_artist_eval = generate_per_artist(final_sample, SYSTEM_PROMPT, "qwen2.5:7b")
pd.DataFrame(qwen_artist_eval).to_json(wd + "qwen_artist_eval.json", orient = "records", indent = 2)

5/100
10/100
15/100
20/100
25/100
30/100
35/100
40/100
45/100
50/100
55/100
60/100
65/100
70/100
75/100
80/100
85/100
90/100
95/100
100/100
105/100
110/100


In [ ]:
manual_assessment = pd.read_excel(wd + "template_assessment.xlsx")

In [ ]:
manual_assessment[["llama_h","mistral_h","qwen_h"]] = manual_assessment[["llama_h","mistral_h","qwen_h"]].fillna("no")
manual_assessment

,Unnamed: 0,Artist,totals,LLAMA,llama_h,MISTRAL,mistral_h,QWEN,qwen_h
0,0,Amon Tobin,0,0,no,0,yes,0,no
1,1,Brad Mehldau,1,1,no,1,no,1,no
2,2,Bad Boys Blue,0,0,no,0,yes,0,no
3,3,The Goo Goo Dolls,4,3,no,3,no,3,no
4,4,Skepta,19,5,no,5,no,5,no
...,...,...,...,...,...,...,...,...,...
105,105,Thievery Corporation,3,3,no,3,no,3,no
106,106,Bob James,0,0,no,0,yes,0,no
107,107,Wes Montgomery,0,0,no,0,no,0,no
108,108,Pedro Infante,2,2,no,1,no,1,no


In [ ]:
hits = manual_assessment[manual_assessment.totals > 0].reset_index(drop = True)
hits.totals = hits.totals.apply(lambda x: 5 if x > 5 else x)
hits["percentage_llama"] = hits["LLAMA"]  / hits["totals"]
hits["percentage_mistral"] = hits["MISTRAL"]  / hits["totals"]
hits["percentage_qwen"] = hits["QWEN"]  / hits["totals"]
table_1 = pd.DataFrame([{"Model" : "LLAMA", "adaptability_score" : hits.percentage_llama.mean(), "hallucination_rate_positive" : list(hits.llama_h).count("yes") / len(hits)},
                     {"Model" : "MISTRAL", "adaptability_score" : hits.percentage_mistral.mean(), "hallucination_rate_positive" : list(hits.mistral_h).count("yes") / len(hits)},
                     {"Model" : "QWEN", "adaptability_score" : hits.percentage_qwen.mean(), "hallucination_rate_positive" : list(hits.qwen_h).count("yes") / len(hits)}])
table_1




,Model,adaptability_score,hallucination_rate_positive
0,LLAMA,0.824561,0.052632
1,MISTRAL,0.931871,0.070175
2,QWEN,0.892105,0.035088


In [ ]:
misses = manual_assessment[manual_assessment.totals == 0].reset_index(drop = True)
table_2 = pd.DataFrame([{"Model" : "LLAMA", "hallucination_rate_negative" : list(misses.llama_h).count("yes") / len(misses)},
                        {"Model" : "MISTRAL", "hallucination_rate_negative" : list(misses.mistral_h).count("yes") / len(misses)},
                        {"Model" : "QWEN", "hallucination_rate_negative" : list(misses.qwen_h).count("yes") / len(misses)}])
table_2

,Model,hallucination_rate_negative
0,LLAMA,0.056604
1,MISTRAL,0.547170
2,QWEN,0.018868


In [ ]:
table_adaptability = table_1.merge(table_2, on = "Model")
table_adaptability = table_adaptability.applymap(lambda x: f"{x * 100:.1f}%" if isinstance(x, (int, float)) else x)
table_adaptability.set_index("Model", inplace = True)
table_adaptability

,adaptability_score,hallucination_rate_positive,hallucination_rate_negative
Model,,,
LLAMA,82.5%,5.3%,5.7%
MISTRAL,93.2%,7.0%,54.7%
QWEN,89.2%,3.5%,1.9%


----

**Response examples:**

In [ ]:
samples_llama = pd.read_json(wd + "llama_eval.json")
samples_mistral = pd.read_json(wd + "mistral_eval.json")
samples_qwen = pd.read_json(wd + "qwen_eval.json")


In [ ]:
i = 20
samples_llama.head()
print(f"Prompt:\n{samples_llama.text[i]}\n")
print(f"Response:\n{samples_llama.output[i]}")

Prompt:
I just left my girlfriend at the airport, and I am not going to see her for 3 months

Response:
I understand that you're going through a tough time, and my goal is to help you find some comfort in music. After reviewing the candidates, I've curated a list of the top 5 songs that I think will resonate with your emotions.

Here are my picks:

1) **(Lost Her Love) On Our Last Date** by Conway Twitty - This country classic speaks directly to your heartache and longing. The lyrics capture the pain of being apart from someone you love, and the song's gentle melody provides a soothing balm for your soul.
2) **I'll Wait for You** by Joe Nichols - This song is a beautiful expression of devotion and commitment. Even though you're physically apart, the singer promises to wait patiently for his loved one to return. The country-pop soundscapes will transport you to a place of hope and longing.
3) **Tender Hearted Baby** by George Hamilton IV - This 1960s country tune has a timeless quality 